In [8]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import load_model
import json
import numpy as np
import os



In [2]:
data_dir = "/home/naseefnf/Projects/MachineSense/backend/uploads/1"

classes = [f for f in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, f))]
num_classes = len(classes)

print(f"Classes found: {classes}")
print(f"Number of classes: {num_classes}")

Classes found: ['excavator', 'truck']
Number of classes: 2


In [3]:
datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.resnet50.preprocess_input,
    validation_split=0.2
)

train_data = datagen.flow_from_directory(
    data_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    subset="training"
)

val_data = datagen.flow_from_directory(
    data_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    subset="validation"
)

print(f"Training samples: {train_data.samples}")
print(f"Validation samples: {val_data.samples}")
print(f"Class indices: {train_data.class_indices}")

Found 18 images belonging to 2 classes.


Found 3 images belonging to 2 classes.
Training samples: 18
Validation samples: 3
Class indices: {'excavator': 0, 'truck': 1}


In [4]:
pip install Pillow


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
base_model = tf.keras.applications.ResNet50(
    include_top = False,
    weights = "imagenet",
    input_tensor = None,
    input_shape = (224, 224, 3)
)

base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
output = Dense(num_classes, activation="softmax")(x)

model = Model(inputs = base_model.input, outputs=output)

print(f"Model built successfully!")
print(f"Output layer neurons: {num_classes}")

E0000 00:00:1778165541.490932    8894 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1778165541.491787    9503 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1778165541.514315    8894 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Model built successfully!
Output layer neurons: 2


In [6]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
history = model.fit(
    train_data,
    epochs=5,
    validation_data=val_data
)

print(f"Training complete!")
print(f"Final accuracy: {history.history['accuracy'][-1]:.2f}")

Epoch 1/5


I0000 00:00:1778165542.909611    8894 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
W0000 00:00:1778165546.671635    9523 cpu_allocator_impl.cc:82] Allocation of 57802752 exceeds 10% of free system memory.
W0000 00:00:1778165546.742610    9523 cpu_allocator_impl.cc:82] Allocation of 59885568 exceeds 10% of free system memory.
W0000 00:00:1778165546.765463    9523 cpu_allocator_impl.cc:82] Allocation of 14450688 exceeds 10% of free system memory.
W0000 00:00:1778165546.778816    9523 cpu_allocator_impl.cc:82] Allocation of 14450688 exceeds 10% of free system memory.
W0000 00:00:1778165546.778820    9526 cpu_allocator_impl.cc:82] Allocation of 57802752 exceeds 10% of free system memory.


1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step - accuracy: 0.3333 - loss: 1.1881 - val_accuracy: 0.6667 - val_loss: 1.9841
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.5556 - loss: 1.9019 - val_accuracy: 0.6667 - val_loss: 0.6564
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 1.0000 - loss: 0.1564 - val_accuracy: 0.6667 - val_loss: 0.3185
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.8889 - loss: 0.2594 - val_accuracy: 0.6667 - val_loss: 0.6372
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.8333 - loss: 0.3870 - val_accuracy: 0.6667 - val_loss: 0.5148
Training complete!
Final accuracy: 0.83


In [7]:
import json
model_save_path = "/home/naseefnf/Projects/MachineSense/backend/trained_models/1_model.h5"
model.save(model_save_path)

label_map = {v: k for k,v in train_data.class_indices.items()}
label_map_path = "/home/naseefnf/Projects/MachineSense/backend/trained_models/1_labels.json"

with open(label_map_path, "w") as f:
    json.dump(label_map, f)

print(f"Model saved to: {model_save_path}")
print(f"Labels saved: {label_map}")

Model saved to: /home/naseefnf/Projects/MachineSense/backend/trained_models/1_model.h5
Labels saved: {0: 'excavator', 1: 'truck'}


In [9]:
loaded_model = load_model("/home/naseefnf/Projects/MachineSense/backend/trained_models/1_model.h5")

with open("/home/naseefnf/Projects/MachineSense/backend/trained_models/1_labels.json") as f:
    label_map = json.load(f)

print("Model loaded successfully!")
print(f"Labels: {label_map}")

Model loaded successfully!
Labels: {'0': 'excavator', '1': 'truck'}


In [14]:
from tensorflow.keras.preprocessing import image

def predict_image(img_path):
    # Load and preprocess image
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = tf.keras.applications.resnet50.preprocess_input(img_array)
    
    # Predict
    predictions = loaded_model.predict(img_array)
    predicted_index = np.argmax(predictions[0])
    confidence = predictions[0][predicted_index] * 100
    machine_name = label_map[str(predicted_index)]
    
    return {
        "machine": machine_name,
        "confidence": round(float(confidence), 2)
    }

In [15]:
# Use any image from your uploads folder
test_image_path = "/home/naseefnf/Projects/MachineSense/backend/uploads/1/truck/test.jpg"

result = predict_image(test_image_path)
print(f"Machine: {result['machine']}")
print(f"Confidence: {result['confidence']}%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
Machine: truck
Confidence: 90.47%
